# Vector Store

A vector store is a system designed to store and retrieve data represented as numerical vectors.

### Key Features
1. Storage – Ensures that vectors and their associated metadata are retained, whether inmemory for quick lookups or on-disk for durability and large-scale use.
2. Similarity Search - Helps retrieve the vectors most similar to a query vector.
3. Indexing - Provide a data structure or method that enables fast similarity searches on
high-dimensional vectors (e.g., approximate nearest neighbor lookups).
4. CRUD Operations - Manage the lifecycle of data—adding new vectors, reading them, updating existing entries, removing outdated vectors.

### Use-cases
1. Semantic Search
2. RAG
3. Recommender Systems
4. Image/Multimedia search

---

**Vector Store**
- Typically refers to a lightweight library or service that focuses on storing vectors (embeddings) and performing similarity search.
- May not include many traditional database features like transactions, rich query languages, or role-based access control.
- Ideal for prototyping, smaller-scale applications
- Examples: FAISS (where you store vectors and can query them by similarity, but you handle persistence and scaling separately).

**Vector Database**
- A full-fledged database system designed to store and query vectors. 
- Offers additional “database-like” features:
    - Distributed architecture for horizontal scaling
    - Durability and persistence (replication, backup/restore)
    - Metadata handling (schemas, filters)
    - Potential for ACID or near-ACID guarantees
    - Authentication/authorization and more advanced security
- Geared for production environments with significant scaling, large datasets
- Examples: Milvus, Qdrant, Weaviate.

A vector database is effectively a vector store with extra database features (e.g.,
clustering, scaling, security, metadata filtering, and durability)

---


## Vector Stores in Langchain

- Supported Stores: LangChain integrates with multiple vector stores (FAISS, Pinecone, Chroma, Qdrant, Weaviate, etc.), giving you flexibility in scale, features, and deployment.

- Common Interface: A uniform Vector Store API lets you swap out one backend (e.g., FAISS) for another (e.g., Pinecone) with minimal code changes.

- Metadata Handling: Most vector stores in LangChain allow you to attach metadata (e.g., timestamps, authors) to each document, enabling filter-based retrieval.

----



## Chroma Vector Store
Chroma is a lightweight, open-source vector database that is especially friendly for local development and small- to medium-scale production needs

![Chroma Vector Store](chroma_vector_store.png)

In [ ]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
  os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter API key for NVIDIA: ")

from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

embeddings = NVIDIAEmbeddings(model="nvidia/llama-nemotron-embed-vl-1b-v2")


c:\Users\Atul.Mangla\Desktop\Learning-Langchain\.venv\lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:243: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [4]:
res = embeddings.embed_query("What is the capital of India?")
print(res)
print(len(res))

[-0.0031833648681640625, 0.021942138671875, 0.01324462890625, 0.035400390625, 0.026458740234375, 0.01136016845703125, -0.05084228515625, 0.01080322265625, -0.026947021484375, -0.0146026611328125, -0.0011205673217773438, -0.011962890625, -0.0229339599609375, -0.0196533203125, 0.0009899139404296875, 0.0233612060546875, -0.005519866943359375, 0.006397247314453125, 0.0129547119140625, 0.005962371826171875, -0.0009469985961914062, 0.0270843505859375, -0.0166778564453125, 0.02264404296875, -0.01221466064453125, 0.0177764892578125, 0.006359100341796875, -0.01995849609375, 0.01824951171875, 0.0194091796875, -0.005157470703125, 0.0247039794921875, -0.024444580078125, 0.002285003662109375, 0.0022907257080078125, 0.029632568359375, -0.0011920928955078125, -0.02105712890625, 0.051422119140625, 0.00763702392578125, -0.022247314453125, 0.0046539306640625, -0.0013799667358398438, -0.00213623046875, -0.0014257431030273438, -0.01328277587890625, -0.032989501953125, 0.00865936279296875, -0.0176391601562

In [6]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [10]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="sample",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

vector_store.add_documents([doc1, doc2, doc3, doc4, doc5])


['e975d7bc-2e5b-4ba9-a53c-8e73da5f384c',
 '3a969f39-af8e-4790-bc95-a9bf9abf7206',
 'f30a1c95-66d4-47a2-a69d-69c15dad56b5',
 'd7958317-9f24-4c37-b3d1-2c93fad5ce3a',
 '277cefc0-7788-4532-956e-d5ad6bfcbef6']

In [9]:
query = "Which team is most successful in IPL history?"
results = vector_store.similarity_search(query, k=2)
for result in results:
    print("Team: {}, Content: {}".format(result.metadata["team"], result.page_content))

Team: Mumbai Indians, Content: Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.
Team: Royal Challengers Bangalore, Content: Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.


In [12]:
query = "Who is greatest Batsman in IPL"
result = vector_store.similarity_search_with_score(query,k=2)
# print(result)
for res in result:
    print("Team: {}\nContent: {}, Score: {}".format(res[0].metadata["team"], res[0].page_content, res[1]))

Team: Royal Challengers Bangalore
Content: Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons., Score: 1.12833833694458
Team: Mumbai Indians
Content: Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure., Score: 1.1752722263336182


In [14]:
vector_store.get(include=['metadatas', 'documents', 'embeddings'])

{'ids': ['e975d7bc-2e5b-4ba9-a53c-8e73da5f384c',
  '3a969f39-af8e-4790-bc95-a9bf9abf7206',
  'f30a1c95-66d4-47a2-a69d-69c15dad56b5',
  'd7958317-9f24-4c37-b3d1-2c93fad5ce3a',
  '277cefc0-7788-4532-956e-d5ad6bfcbef6'],
 'embeddings': array([[ 0.01783752, -0.04653931,  0.02006531, ...,  0.00886536,
         -0.0149765 , -0.04052734],
        [-0.00116348, -0.07800293, -0.00956726, ...,  0.00282097,
         -0.0086441 , -0.00252151],
        [ 0.01435852, -0.04431152,  0.014328  , ...,  0.00476074,
         -0.02513123,  0.00861359],
        [ 0.01395416, -0.05212402,  0.01246643, ...,  0.00218391,
         -0.04040527, -0.01637268],
        [ 0.02087402, -0.0574646 ,  0.02645874, ..., -0.01366425,
         -0.00224495, -0.03701782]], shape=(5, 2048)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m